In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

TIME_OFFSET = 10800 #ADT to UTC is 3hrs

In [2]:
def entropy_feature(x):
    counts = x.value_counts()
    if len(counts) <= 1:
        return 0.0
    return entropy(counts)

def coeff_var(x):
    mean = x.mean()
    if mean == 0:
        return 0.0
    return x.std() / mean

In [3]:
import pandas as pd
import numpy as np

def process_file(k, input_csv, output_csv):
    print(f"Processing {input_csv}...")

    df = pd.read_csv(input_csv)

    # ── Type coercion ──────────────────────────────────────────────────────────
    numeric_cols = [
        "tcp.len", "mbtcp.len", "modbus.byte_cnt", "modbus.regval_uint16",
        "modbus.reference_num", "modbus.response_time", "modbus.word_cnt",
        "tcp.analysis.retransmission", "modbus.func_code", "modbus.exception_code",
        "mbtcp.unit_id", "mbtcp.trans_id",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── Drop non-IP / malformed rows ───────────────────────────────────────────
    df = df.dropna(subset=["ip.src", "ip.dst", "tcp.len"])

    # ── True zero fills ────────────────────────────────────────────────────────
    df["tcp.analysis.retransmission"] = df["tcp.analysis.retransmission"].fillna(0).astype(int)
    df["modbus.exception_code"]       = df["modbus.exception_code"].fillna(0)
    df["modbus.response_time"]        = df["modbus.response_time"].fillna(0)
    df["modbus.request_frame"]        = df["modbus.request_frame"].fillna(0)

    # ── FC-conditional fills (safe to 0; only aggregate on relevant FC subsets) 
    df["modbus.reference_num"]  = df["modbus.reference_num"].fillna(0)
    df["modbus.regnum16"]       = df["modbus.regnum16"].fillna(0)
    df["modbus.regval_uint16"]  = df["modbus.regval_uint16"].fillna(0)
    df["modbus.bitnum"]         = df["modbus.bitnum"].fillna(0)
    df["modbus.bit_cnt"]        = df["modbus.bit_cnt"].fillna(0)
    df["modbus.word_cnt"]       = df["modbus.word_cnt"].fillna(0)
    df["modbus.byte_cnt"]       = df["modbus.byte_cnt"].fillna(0)
    df["modbus.data"]           = df["modbus.data"].fillna("")

    # ── Precompute derived columns on Modbus rows only ─────────────────────────
    # mbtcp.len is NaN for non-Modbus packets; derived columns default to 0
    modbus_mask = df["modbus.func_code"].notna()
    df["is_stacked"]   = 0.0
    df["stack_excess"] = 0.0
    df["len_mismatch"] = 0.0
    df["is_mismatch"]  = False

    df.loc[modbus_mask, "is_stacked"]   = (
        df.loc[modbus_mask, "tcp.len"] > (df.loc[modbus_mask, "mbtcp.len"] + 6)
    ).astype(float)
    df.loc[modbus_mask, "stack_excess"] = (
        df.loc[modbus_mask, "tcp.len"] - (df.loc[modbus_mask, "mbtcp.len"] + 6)
    ).clip(lower=0)
    df.loc[modbus_mask, "len_mismatch"] = (
        df.loc[modbus_mask, "mbtcp.len"] - df.loc[modbus_mask, "modbus.byte_cnt"]
    )
    df.loc[modbus_mask, "is_mismatch"]  = (
        df.loc[modbus_mask, "len_mismatch"].abs() > 2
    )

    # is_duplicate runs on full df; NaN in mbtcp.trans_id/modbus.data treated as
    # a distinct value by duplicated() so non-Modbus rows won't spuriously match
    df["is_duplicate"] = df.duplicated(
        subset=["ip.src", "ip.dst", "mbtcp.trans_id", "modbus.data"], keep=False
    )

    # ── Timestamp & windowing ──────────────────────────────────────────────────
    df["timestamp"]   = df["frame.time_epoch"] + TIME_OFFSET
    df = df.sort_values("timestamp")
    df["time_window"] = (df["timestamp"] * k).astype(int)

    has_transaction_id = "mbtcp.trans_id" in df.columns

    # ── IAT within (time_window, src, dst) ────────────────────────────────────
    df["iat"] = (
        df.groupby(["time_window", "ip.src", "ip.dst"])["timestamp"]
        .diff()
        .fillna(0)
    )

    temp_src  = df["ip.src"].dropna().astype(str)
    temp_dst  = df["ip.dst"].dropna().astype(str)
    known_ips = sorted(set(temp_src.unique()) | set(temp_dst.unique()))
    print(f"IPs found: {known_ips}")

    observed_windows = pd.Index(sorted(df["time_window"].unique()), name="time_window")

    # ── Helper aggregation functions ───────────────────────────────────────────
    def coeff_var(x):
        m = x.mean()
        return x.std() / m if m > 0 else 0.0

    def entropy_feature(x):
        counts = x.value_counts(normalize=True)
        return -(counts * np.log2(counts + 1e-9)).sum()

    # ── Per-IP feature extraction ──────────────────────────────────────────────
    ip_frames = []

    for ip in known_ips:
        tx = df[df["ip.src"] == ip]
        rx = df[df["ip.dst"] == ip]

        tx_agg = tx.groupby("time_window").agg(
            # Volume
            tx_packet_count       = ("frame.len",                    "count"),
            tx_total_bytes        = ("frame.len",                    "sum"),
            tx_mean_pkt_size      = ("frame.len",                    "mean"),
            tx_std_pkt_size       = ("frame.len",                    "std"),
            # IAT
            tx_iat_mean           = ("iat",                          "mean"),
            tx_iat_std            = ("iat",                          "std"),
            tx_iat_cv             = ("iat",                          coeff_var),
            # Destinations & slaves
            tx_unique_dst         = ("ip.dst",                       "nunique"),
            tx_unique_unit_ids    = ("mbtcp.unit_id",                "nunique"),
            # Function codes
            tx_unique_fc          = ("modbus.func_code",             "nunique"),
            tx_func_entropy       = ("modbus.func_code",             entropy_feature),
            tx_read_count         = ("modbus.func_code",             lambda x: x.isin([3, 4]).sum()),
            tx_write_count        = ("modbus.func_code",             lambda x: x.isin([5, 6, 15, 16]).sum()),
            tx_fc3                = ("modbus.func_code",             lambda x: (x == 3).sum()),
            tx_fc4                = ("modbus.func_code",             lambda x: (x == 4).sum()),
            tx_fc5                = ("modbus.func_code",             lambda x: (x == 5).sum()),
            tx_fc6                = ("modbus.func_code",             lambda x: (x == 6).sum()),
            tx_fc15               = ("modbus.func_code",             lambda x: (x == 15).sum()),
            tx_fc16               = ("modbus.func_code",             lambda x: (x == 16).sum()),
            # Exceptions
            tx_exception_count    = ("modbus.exception_code",        lambda x: x.notna().sum()),
            # Register addressing
            tx_unique_regs        = ("modbus.reference_num",         "nunique"),
            tx_register_std       = ("modbus.reference_num",         "std"),
            tx_register_entropy   = ("modbus.reference_num",         entropy_feature),
            # Register values
            tx_regval_mean        = ("modbus.regval_uint16",         "mean"),
            tx_regval_std         = ("modbus.regval_uint16",         "std"),
            # Payload sizes
            tx_mean_mbap_len      = ("mbtcp.len",                    "mean"),
            tx_byte_cnt_mean      = ("modbus.byte_cnt",              "mean"),
            tx_word_cnt_mean      = ("modbus.word_cnt",              "mean"),
            # Response timing
            tx_resp_time_mean     = ("modbus.response_time",         "mean"),
            tx_resp_time_std      = ("modbus.response_time",         "std"),
            # Retransmissions
            tx_retransmission_sum = ("tcp.analysis.retransmission",  "sum"),
            # Stacking
            tx_stacked_count      = ("is_stacked",                   "sum"),
            tx_mean_stack_excess  = ("stack_excess",                 "mean"),
            # Length mismatch
            tx_len_mismatch_mean  = ("len_mismatch",                 "mean"),
            tx_len_mismatch_count = ("is_mismatch",                  "sum"),
            # Duplicate / replay
            tx_duplicate_count    = ("is_duplicate",                 "sum"),
        )

        rx_agg = rx.groupby("time_window").agg(
            rx_packet_count       = ("frame.len",                    "count"),
            rx_total_bytes        = ("frame.len",                    "sum"),
            rx_unique_src         = ("ip.src",                       "nunique"),
            rx_unique_unit_ids    = ("mbtcp.unit_id",                "nunique"),
            rx_retransmission_sum = ("tcp.analysis.retransmission",  "sum"),
            rx_resp_time_mean     = ("modbus.response_time",         "mean"),
        )

        if has_transaction_id:
            tx_tid = tx.groupby("time_window").agg(
                tx_unique_tids = ("mbtcp.trans_id", "nunique")
            )
            tx_agg = tx_agg.join(tx_tid, how="left")

        tx_agg = tx_agg.reindex(observed_windows, fill_value=0)
        rx_agg = rx_agg.reindex(observed_windows, fill_value=0)

        ip_df = tx_agg.join(rx_agg, how="left").fillna(0)

        # ── Derived ratios ─────────────────────────────────────────────────────
        pkt = ip_df["tx_packet_count"].to_numpy(dtype=float)

        def safe_div(num_col):
            num = ip_df[num_col].to_numpy(dtype=float)
            return np.divide(num, pkt, out=np.zeros_like(pkt), where=pkt > 0)

        ip_df["tx_write_ratio"]     = safe_div("tx_write_count")
        ip_df["tx_read_ratio"]      = safe_div("tx_read_count")
        ip_df["tx_exception_ratio"] = safe_div("tx_exception_count")

        if has_transaction_id:
            ip_df["tx_dup_tids"]  = ip_df["tx_packet_count"] - ip_df["tx_unique_tids"]
            ip_df["tx_tid_ratio"] = safe_div("tx_unique_tids")
        else:
            ip_df["tx_unique_tids"] = 0
            ip_df["tx_dup_tids"]    = 0
            ip_df["tx_tid_ratio"]   = 0.0

        ip_df["tx_regval_delta"] = ip_df["tx_regval_mean"].diff().fillna(0)

        # ── Prefix with IP ─────────────────────────────────────────────────────
        safe_ip = ip.replace(".", "_")
        ip_df   = ip_df.add_prefix(f"{safe_ip}__")

        ip_frames.append(ip_df)

    # ── Join all IPs into one row per time_window ──────────────────────────────
    result = pd.DataFrame(index=observed_windows)
    for ip_df in ip_frames:
        result = result.join(ip_df, how="left")

    result = result.fillna(0).reset_index()

    packet_counts = df.groupby("time_window").size()
    print("Window packet count description:")
    print(packet_counts.describe())

    result.to_csv(output_csv, index=False)
    print(f"Saved → {output_csv}\n")

In [4]:
process_file(1, "../train/benign_nw_analysis.csv", "../train/1s_benign_flows.csv")

Processing ../train/benign_nw_analysis.csv...
IPs found: ['185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.8']


/tmp/ipykernel_670463/4246770210.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count    7069.000000
mean       75.930966
std        90.016900
min         1.000000
25%        50.000000
50%        50.000000
75%        90.000000
max      1291.000000
dtype: float64
Saved → ../train/1s_benign_flows.csv



In [5]:
process_file(1, "../train/cscada_attack_ssw_analysis.csv", "../train/1s_cscada_flows.csv")

Processing ../train/cscada_attack_ssw_analysis.csv...


/tmp/ipykernel_670463/4246770210.py:7: DtypeWarning: Columns (0: mbtcp.trans_id, 1: mbtcp.unit_id, 2: mbtcp.len, 3: mbtcp.prot_id, 4: modbus.func_code, 5: modbus.reference_num, 6: modbus.regnum16, 7: modbus.bitnum, 8: modbus.word_cnt, 9: modbus.bit_cnt, 10: modbus.regval_uint16, 11: modbus.data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


IPs found: ['185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.8']


/tmp/ipykernel_670463/4246770210.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count    14960.000000
mean        77.215775
std        144.355317
min          1.000000
25%         45.000000
50%         50.000000
75%         90.000000
max       8241.000000
dtype: float64
Saved → ../train/1s_cscada_flows.csv



In [6]:
process_file(1, "../train/ext_attack_nw_analysis.csv", "../train/1s_external_flows.csv")

Processing ../train/ext_attack_nw_analysis.csv...


/tmp/ipykernel_670463/4246770210.py:7: DtypeWarning: Columns (0: modbus.data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


IPs found: ['185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.7', '185.175.0.8']


/tmp/ipykernel_670463/4246770210.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count     1877.000000
mean       291.241343
std       1253.904741
min          1.000000
25%         45.000000
50%         45.000000
75%         51.000000
max      20281.000000
dtype: float64
Saved → ../train/1s_external_flows.csv



In [7]:
df = pd.read_csv("../train/1s_cscada_flows.csv")
print(df.columns)

Index(['time_window', '185_175_0_3__tx_packet_count',
       '185_175_0_3__tx_total_bytes', '185_175_0_3__tx_mean_pkt_size',
       '185_175_0_3__tx_std_pkt_size', '185_175_0_3__tx_iat_mean',
       '185_175_0_3__tx_iat_std', '185_175_0_3__tx_iat_cv',
       '185_175_0_3__tx_unique_dst', '185_175_0_3__tx_unique_unit_ids',
       ...
       '185_175_0_8__rx_unique_src', '185_175_0_8__rx_unique_unit_ids',
       '185_175_0_8__rx_retransmission_sum', '185_175_0_8__rx_resp_time_mean',
       '185_175_0_8__tx_write_ratio', '185_175_0_8__tx_read_ratio',
       '185_175_0_8__tx_exception_ratio', '185_175_0_8__tx_dup_tids',
       '185_175_0_8__tx_tid_ratio', '185_175_0_8__tx_regval_delta'],
      dtype='str', length=246)


In [8]:
# process_file(10, "../train/benign_nw.csv", "../train/100ms_benign_flows.csv") #this is for one second windows

In [9]:
# process_file(10, "../train/cscada_attack_ssw.csv", "../train/100ms_cscada_flows.csv")

In [10]:
# process_file(10, "../train/ext_attack_nw.csv", "../train/100ms_external_flows.csv")

In [11]:
import os
os.system('notify-send "Python Script" "Execution complete!"')

0